# Chapter 11 - Neural Networks that Understand Language

## Summary
Chapter 11 applies neural networks to **natural language**. Words are not numbers, so we must first *encode* them.
- **Bag-of-words input:** a review becomes a vector with one slot per vocabulary word (1 if the word appears). The network is trained to predict *positive vs. negative* sentiment.
- **Correlation again:** the network learns that some words correlate with positive reviews and others with negative ones. Looking at the learned weights shows which words matter ("what the network learned").
- **Embedding layer:** instead of multiplying a huge sparse vector by a matrix, we simply **sum (or average) the weight rows of the words present** - mathematically identical to one-hot x matrix, but much faster.
- **Word embeddings:** words that occur in similar contexts get *similar vectors*. Words are then comparable by distance, and even **analogies** work (`king - man + woman ~ queen`).
- **Neural architecture matters:** how you encode the input decides what the network can find.
- **Filling in the blank / noise handling (book):** predicting a hidden word from the surrounding words forces useful embeddings (the idea behind word2vec).

## Theory
- **Why one-hot is wasteful:** `one_hot (1 x V) . W (V x H)` just selects the rows of `W` for words that are 'on'. Index-select + sum avoids the multiplication.
- **Embedding:** each word owns a learnable row of `W`. Rows are tuned by gradient descent, so word meaning is encoded as a position in space.
- **Distributional hypothesis:** *"you shall know a word by the company it keeps"* - similar contexts -> similar vectors.
- **Vector arithmetic:** directions in embedding space encode concepts (gender, royalty). `king - man + woman` removes the 'male' direction and adds 'female', landing near `queen`.
- **Similarity:** Euclidean distance (or cosine) between vectors.
- **Note on data:** the book uses 25,000 IMDB reviews. To stay offline and fast, this notebook uses a small *generated* review corpus; the algorithm is identical.

In [1]:
# If packages are missing in this kernel, uncomment:
# %pip install numpy scikit-learn matplotlib
import numpy as np

### 1) A small movie-review corpus (generated) and bag-of-words sentiment network

In [2]:
np.random.seed(1)
pos_words = ["great", "wonderful", "amazing", "brilliant", "enjoyable", "superb", "loved"]
neg_words = ["terrible", "boring", "awful", "dull", "horrible", "bad", "hated"]
neutral   = ["movie", "film", "story", "actor", "plot", "scene", "the", "a", "this", "was", "and", "very", "ending", "music"]

def make_review(positive):
    n = np.random.randint(6, 12)
    words = list(np.random.choice(neutral, n))
    main, other = (pos_words, neg_words) if positive else (neg_words, pos_words)
    words += list(np.random.choice(main, np.random.randint(1, 4)))
    if np.random.rand() < 0.15: words.append(np.random.choice(other))   # label noise
    np.random.shuffle(words)
    return words, int(positive)

data = [make_review(i % 2 == 0) for i in range(2000)]
train, test = data[:1600], data[1600:]
vocab = sorted({w for r, _ in data for w in r})
word2index = {w: i for i, w in enumerate(vocab)}
print(len(vocab), "words; example:", " ".join(train[0][0]), "->", train[0][1])

def to_indices(review): return list({word2index[w] for w in review})
train_x = [to_indices(r) for r, _ in train]; train_y = np.array([l for _, l in train])
test_x  = [to_indices(r) for r, _ in test];  test_y  = np.array([l for _, l in test])

28 words; example: movie was enjoyable film very movie ending ending a loved this scene very -> 1


In [3]:
# Embedding-style forward pass: sum the weight rows of words present (== one-hot . W)
def sigmoid(x): return 1 / (1 + np.exp(-x))
hidden, alpha, iters = 16, 0.01, 6
w01 = 0.2 * np.random.random((len(vocab), hidden)) - 0.1
w12 = 0.2 * np.random.random((hidden, 1)) - 0.1

for it in range(iters):
    correct = 0
    for x, y in zip(train_x, train_y):
        l1 = sigmoid(np.sum(w01[x], axis=0, keepdims=True))   # embedding sum
        l2 = sigmoid(l1.dot(w12))
        d2 = l2 - y
        d1 = d2.dot(w12.T) * l1 * (1 - l1)
        w12 -= alpha * l1.T.dot(d2)
        w01[x] -= alpha * d1                                   # update only rows used
        correct += abs(d2) < 0.5
    tc = sum(abs(sigmoid(sigmoid(np.sum(w01[x], 0, keepdims=True)).dot(w12)) - y) < 0.5 for x, y in zip(test_x, test_y))
    print(f"iter {it}  train-acc {correct[0][0]/len(train_x):.3f}  test-acc {tc[0][0]/len(test_x):.3f}")

iter 0  train-acc 0.406  test-acc 0.853
iter 1  train-acc 0.814  test-acc 0.965
iter 2  train-acc 0.964  test-acc 0.975
iter 3  train-acc 0.972  test-acc 0.978
iter 4  train-acc 0.975  test-acc 0.978
iter 5  train-acc 0.977  test-acc 0.978


### 2) What did the network learn? Words most similar to a target word (by weight vector distance)

In [4]:
def similar(target, k=5):
    v = w01[word2index[target]]
    dist = ((w01 - v) ** 2).sum(1)
    return [str(vocab[i]) for i in np.argsort(dist)[:k]]
for w in ["great", "terrible", "movie"]:
    print(w, "->", similar(w))

great -> ['great', 'wonderful', 'loved', 'enjoyable', 'superb']
terrible -> ['terrible', 'awful', 'dull', 'horrible', 'hated']
movie -> ['movie', 'story', 'music', 'was', 'scene']


Positive words cluster near other positive words and negative words near negative ones, because they have **similar correlation with the label**. Neutral words (`movie`, `film`) sit apart - they carry little signal.

### 2b) Word analogies: `king - man + woman = ?`
A tiny hand-made corpus encodes gender and royalty. We build word vectors from **co-occurrence counts + PPMI + SVD** (a count-based cousin of word2vec).

In [5]:
sentences = []
male_ctx   = ["he", "his", "beard", "sir"]
female_ctx = ["she", "her", "dress", "madam"]
royal_ctx  = ["crown", "throne", "palace", "rules"]
common_ctx = ["lives", "works", "walks", "talks"]
adult_ctx  = ["old", "job", "wise", "tax"]
young_ctx  = ["school", "toys", "play", "homework"]
people = {"man": (1,0), "woman": (0,1), "king": (1,0), "queen": (0,1), "boy": (1,0), "girl": (0,1),
          "prince": (1,0), "princess": (0,1)}
royal = {"king", "queen", "prince", "princess"}
young = {"boy", "girl", "prince", "princess"}
rng = np.random.RandomState(0)
for p, (m, f) in people.items():
    for _ in range(60):
        ctx = list(rng.choice(common_ctx, 2))
        ctx += list(rng.choice(male_ctx if m else female_ctx, 2))
        ctx += list(rng.choice(young_ctx if p in young else adult_ctx, 2))
        if p in royal: ctx += list(rng.choice(royal_ctx, 2))
        sentences.append([p] + ctx)

words = sorted({w for s in sentences for w in s}); idx = {w: i for i, w in enumerate(words)}
C = np.zeros((len(words), len(words)))
for s in sentences:
    for a in s:
        for b in s:
            if a != b: C[idx[a], idx[b]] += 1
P = C / C.sum(); pw = P.sum(1, keepdims=True); pc = P.sum(0, keepdims=True)
ppmi = np.maximum(np.log((P + 1e-12) / (pw * pc + 1e-12)), 0)
U, S, _ = np.linalg.svd(ppmi)
emb = U[:, :8] * S[:8]
emb /= np.linalg.norm(emb, axis=1, keepdims=True)

def analogy(a, b, c, exclude=True):
    v = emb[idx[a]] - emb[idx[b]] + emb[idx[c]]
    sims = emb.dot(v / np.linalg.norm(v))
    order = [str(words[i]) for i in np.argsort(-sims) if words[i] not in (a, b, c)]
    return order[:3]
print("king - man + woman     ->", analogy("king", "man", "woman"))
print("prince - boy + girl    ->", analogy("prince", "boy", "girl"))
print("queen - woman + man    ->", analogy("queen", "woman", "man"))

king - man + woman     -> ['queen', 'lives', 'princess']
prince - boy + girl    -> ['princess', 'queen', 'lives']
queen - woman + man    -> ['king', 'prince', 'walks']


## Takeaways
- Language is made learnable by an **embedding layer** - a learnable lookup table.
- Words with similar roles end up with similar vectors, so meaning becomes geometry (distance, analogy).
- Using index-select + sum instead of one-hot multiplication gives the same result far more cheaply.